In [ ]:
# Cell 1: imports
import json, os, random
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image, ImageDraw
import seaborn as sns
sns.set()
R = Path("data")


In [ ]:
# Cell 2: load metadata
csv = pd.read_csv("data/raw/annotations/dataset.csv")
coco = json.load(open("data/raw/annotations/COCO_fracture_masks.json"))
images = {img['file_name']: img for img in coco['images']}
annotations = coco['annotations']


In [ ]:
# Cell 3: summary
print("CSV rows:", len(csv))
print("COCO images:", len(coco['images']))
print("COCO annotations:", len(annotations))
display(csv.head())


In [ ]:
# Cell 4: fractured distribution and region counts
fig, axes = plt.subplots(1,2, figsize=(12,4))
csv['fractured'].value_counts().plot.pie(ax=axes[0], autopct='%1.1f%%', title='fractured')
csv[['hand','leg','hip','shoulder','mixed']].sum().sort_values().plot.bar(ax=axes[1], title='region counts')
plt.tight_layout()


In [ ]:
# Cell 5: view and hardware
print(csv[['frontal','lateral','oblique']].sum())
print("hardware present:", csv['hardware'].sum())


In [ ]:
# Cell 6: show random annotated images
def draw_boxes(img_path, anns_for_img):
    im = Image.open(img_path).convert("RGB")
    draw = ImageDraw.Draw(im)
    for a in anns_for_img:
        x,y,w,h = a['bbox']
        draw.rectangle([x,y,x+w,y+h], outline="red", width=3)
    return im

sample_files = random.sample(list(images.keys()), 6)
for f in sample_files:
    img_meta = images[f]
    anns = [a for a in annotations if a['image_id']==img_meta['id']]
    display(draw_boxes(f"data/raw/images/{f}", anns))


In [ ]:
# Cell 7: basic validation
missing = [f for f in images if not Path(f"data/raw/images/{f}").exists()]
print("missing files:", len(missing))
# bbox bounds check
errs = []
for a in annotations:
    img = next(i for i in coco['images'] if i['id']==a['image_id'])
    x,y,w,h = a['bbox']
    if x<0 or y<0 or x+w>img['width']+1 or y+h>img['height']+1:
        errs.append(a['id'])
print("bbox OOB count:", len(errs))


In [ ]:
# Cell 8: save summary CSV
summary = {
  "n_images": len(coco['images']),
  "n_annotations": len(annotations),
  "fractured_ratio": csv['fractured'].mean()
}
pd.Series(summary).to_frame("value").to_csv("experiments/dataset_summary.csv")
print("saved experiments/dataset_summary.csv")
